# Renger 2026 Local Parameter Inference

**Objective**
- Infer local `EC`, `EJmax`, `flux`, and `asymmetry` priors for `QB1`, `TC1`, `QB2`, and `TC2` using only single-device circuit Hamiltonians.
- Freeze one reproducible `paper_local_priors_v1` snapshot for later coupled-model work.

**Success criteria**
- `QB1` and `QB2` match the Table II `f01` targets within `20 MHz` and the derived `EC` targets within `5 MHz`.
- `TC1` and `TC2` reproduce the Appendix C coupler anharmonicity prior `alpha_c ~= -0.11 GHz` while remaining explicitly labeled weakly identified.
- The notebook writes `output/renger2026/paper_local_priors.toml` and demonstrates uncoupled `FluxControl` sanity checks.


## Outline

1. Activate the repository environment and load the current `QuantumCircuit` API.
2. Encode the paper targets from Table II, Table I, and Appendix C.
3. Build notebook-local forward models with `CircuitHamiltonianSpec`.
4. Run a deterministic coarse-to-fine search for the two qubits and two couplers.
5. Inspect residuals, profile scans, and fit identifiability.
6. Freeze one accepted `paper_local_priors_v1` snapshot.
7. Run uncoupled `FluxControl` sanity checks on the inferred devices.


In [2]:
using Pkg

function find_repo_root(start::AbstractString)
    candidates = [
        normpath(start),
        normpath(joinpath(start, "..")),
        normpath(joinpath(start, "..", "..")),
    ]

    for candidate in unique(candidates)
        project_toml = joinpath(candidate, "Project.toml")
        if isfile(project_toml)
            content = read(project_toml, String)
            occursin("QuantumCircuit", content) && return candidate
        end
    end

    error("Could not find the QuantumCircuit.jl project root.")
end

project_root = find_repo_root(pwd())
Pkg.activate(project_root)
Pkg.instantiate()

using QuantumCircuit
using Printf
using TOML
using UnicodePlots: lineplot

project_root


  Activating project at `~/Research/20_Projects/QuantumCircuit.jl`


"/Users/yalgaeahn/Research/20_Projects/QuantumCircuit.jl/"

## Step 1 - Paper targets and modeling assumptions

This task stays local: one tunable subsystem at a time, one exact circuit Hamiltonian at a time, and no coupled-device fitting yet.

Modeling choices for this notebook:

- Use exact single-device spectra from `CircuitHamiltonianSpec(charge_cutoff = 10)`.
- Keep the resonator fixed at `4.22 GHz`; it is recorded as context, not inferred here.
- Prefer Table II over Appendix C if the sources disagree.
- Use strong sweet-spot priors at `flux ~= 0` because the parked operating points are first-order flux insensitive.
- Treat qubits as identifiable enough to freeze and couplers as only weakly identified until coupled avoided-crossing data is added.


In [3]:
paper_targets = Dict(
    :QB1 => (
        kind = :qubit,
        name = :qb1,
        target_f01 = 4.67,
        target_EJ = 14.8,
        target_EC = 14.8 / 74.2,
        asymmetry_prior = 0.10,
        ncut = 15,
        charge_cutoff = 10,
    ),
    :QB2 => (
        kind = :qubit,
        name = :qb2,
        target_f01 = 4.47,
        target_EJ = 13.8,
        target_EC = 13.8 / 68.8,
        asymmetry_prior = 0.10,
        ncut = 15,
        charge_cutoff = 10,
    ),
    :TC1 => (
        kind = :coupler,
        name = :tc1,
        target_alpha = -0.11,
        plausible_f01 = (5.0, 8.5),
        asymmetry_prior = 0.10,
        ncut = 13,
        charge_cutoff = 10,
    ),
    :TC2 => (
        kind = :coupler,
        name = :tc2,
        target_alpha = -0.11,
        plausible_f01 = (5.0, 8.5),
        asymmetry_prior = 0.10,
        ncut = 13,
        charge_cutoff = 10,
    ),
)

secondary_constraints = (
    resonator_f01 = 4.22,
    beta_qc_qb1 = 0.017,
    beta_qc_qb2 = 0.022,
    beta_cr = 0.0225,
    beta_qr = 0.0020,
    coupler_alpha_prior = -0.11,
)

target_summary = [
    (
        device = String(label),
        kind = String(target.kind),
        target_f01_ghz = get(target, :target_f01, missing),
        target_alpha_ghz = get(target, :target_alpha, missing),
        target_EJ_ghz = get(target, :target_EJ, missing),
        target_EC_ghz = get(target, :target_EC, missing),
    ) for (label, target) in pairs(paper_targets)
]

sort!(target_summary, by = row -> row.device)
target_summary


4-element Vector{NamedTuple{(:device, :kind, :target_f01_ghz, :target_alpha_ghz, :target_EJ_ghz, :target_EC_ghz)}}:
 (device = "QB1", kind = "qubit", target_f01_ghz = 4.67, target_alpha_ghz = missing, target_EJ_ghz = 14.8, target_EC_ghz = 0.19946091644204852)
 (device = "QB2", kind = "qubit", target_f01_ghz = 4.47, target_alpha_ghz = missing, target_EJ_ghz = 13.8, target_EC_ghz = 0.20058139534883723)
 (device = "TC1", kind = "coupler", target_f01_ghz = missing, target_alpha_ghz = -0.11, target_EJ_ghz = missing, target_EC_ghz = missing)
 (device = "TC2", kind = "coupler", target_f01_ghz = missing, target_alpha_ghz = -0.11, target_EJ_ghz = missing, target_EC_ghz = missing)

## Step 2 - Notebook-local forward model and diagnostics

Every candidate parameter set is scored from exact single-device spectra. The diagnostic bundle includes:

- `f01`
- anharmonicity `alpha`
- parked effective Josephson energy `EJ_eff`
- numerical first-order flux sensitivity `df01/dflux`


In [4]:
effective_josephson_energy(EJmax, flux, asymmetry) =
    EJmax * sqrt(cospi(flux)^2 + (asymmetry * sinpi(flux))^2)

circuit_spec(target) = CircuitHamiltonianSpec(charge_cutoff = target.charge_cutoff)

function build_local_device(target, params)
    if target.kind == :qubit
        return TunableTransmon(
            target.name;
            EJmax = params.EJmax,
            EC = params.EC,
            flux = params.flux,
            asymmetry = params.asymmetry,
            ng = 0.0,
            ncut = target.ncut,
        )
    elseif target.kind == :coupler
        return TunableCoupler(
            target.name;
            EJmax = params.EJmax,
            EC = params.EC,
            flux = params.flux,
            asymmetry = params.asymmetry,
            ng = 0.0,
            ncut = target.ncut,
        )
    end

    error("Unsupported local target kind $(target.kind).")
end

function local_metrics(target, params; levels = 4)
    device = build_local_device(target, params)
    system = CompositeSystem(device)
    spec = spectrum(system; levels = levels, hamiltonian_spec = circuit_spec(target))
    freqs = transition_frequencies(spec)
    return (
        f01 = freqs[1],
        alpha = anharmonicity(spec),
        effEJ = effective_josephson_energy(params.EJmax, params.flux, params.asymmetry),
    )
end

function flux_slope(target, params; delta = 1e-4)
    plus = local_metrics(target, merge(params, (; flux = params.flux + delta))).f01
    minus = local_metrics(target, merge(params, (; flux = params.flux - delta))).f01
    return (plus - minus) / (2 * delta)
end

function base_params(row)
    return (
        EJmax = row.EJmax,
        EC = row.EC,
        flux = row.flux,
        asymmetry = row.asymmetry,
    )
end

local_metrics(paper_targets[:QB1], (EJmax = 14.9, EC = paper_targets[:QB1].target_EC, flux = 0.0, asymmetry = 0.10))


(f01 = 4.667382162451687, alpha = -0.22227159778090488, effEJ = 14.9)

## Step 3 - Deterministic loss functions and search strategy

The search is intentionally simple and dependency-free:

1. Coarse grid over `flux` and `asymmetry`
2. Inner exact scan over `EC` and `EJmax` for each coarse point
3. Multistart coordinate descent over all four parameters

The qubit loss prioritizes `f01`, derived `EC`, and a broad parked `EJ_eff` target. The coupler loss prioritizes `alpha_c ~= -0.11 GHz`, sweet-spot parking, and only a broad parked-frequency plausibility window.


In [5]:
function qubit_loss(target, params; include_slope = false)
    metrics = local_metrics(target, params)
    slope = include_slope ? flux_slope(target, params) : 0.0

    loss = ((metrics.f01 - target.target_f01) / 0.005)^2 +
           ((params.EC - target.target_EC) / 0.002)^2 +
           ((metrics.effEJ - target.target_EJ) / 0.20)^2 +
           (params.flux / 0.02)^2 +
           ((params.asymmetry - target.asymmetry_prior) / 0.08)^2

    include_slope && (loss += (slope / 0.10)^2)

    return merge(params, metrics, (; loss, slope))
end

function coupler_loss(target, params; include_slope = false)
    metrics = local_metrics(target, params)
    slope = include_slope ? flux_slope(target, params) : 0.0
    fmin, fmax = target.plausible_f01

    f_penalty = if metrics.f01 < fmin
        ((fmin - metrics.f01) / 0.5)^2
    elseif metrics.f01 > fmax
        ((metrics.f01 - fmax) / 0.75)^2
    else
        0.0
    end

    loss = ((metrics.alpha - target.target_alpha) / 0.01)^2 +
           f_penalty +
           (params.flux / 0.02)^2 +
           ((params.asymmetry - target.asymmetry_prior) / 0.08)^2

    include_slope && (loss += (slope / 0.10)^2)

    return merge(params, metrics, (; loss, slope))
end

evaluate_candidate(target, params; include_slope = false) =
    target.kind == :qubit ? qubit_loss(target, params; include_slope = include_slope) :
    coupler_loss(target, params; include_slope = include_slope)

function update_topk!(rows, candidate; k = 5)
    push!(rows, candidate)
    sort!(rows, by = row -> row.loss)
    length(rows) > k && resize!(rows, k)
    return rows
end

function parameter_bounds(target)
    if target.kind == :qubit
        return (
            EJmax = (target.target_EJ - 0.8, target.target_EJ + 0.8),
            EC = (target.target_EC - 0.008, target.target_EC + 0.008),
            flux = (-0.10, 0.10),
            asymmetry = (0.0, 0.5),
        )
    end

    return (
        EJmax = (28.0, 46.0),
        EC = (0.090, 0.125),
        flux = (-0.10, 0.10),
        asymmetry = (0.0, 0.5),
    )
end

function coarse_grids(target)
    if target.kind == :qubit
        return (
            flux = collect(-0.04:0.02:0.04),
            asymmetry = collect(0.0:0.05:0.20),
            EC = collect(target.target_EC .+ (-0.004:0.002:0.004)),
            EJmax = collect(target.target_EJ .+ (-0.50:0.05:0.50)),
        )
    end

    return (
        flux = collect(-0.04:0.02:0.04),
        asymmetry = collect(0.0:0.05:0.20),
        EC = collect(0.095:0.0025:0.115),
        EJmax = collect(30.0:1.0:40.0),
    )
end

function set_param(params, field::Symbol, value)
    field == :EJmax && return merge(params, (; EJmax = value))
    field == :EC && return merge(params, (; EC = value))
    field == :flux && return merge(params, (; flux = value))
    field == :asymmetry && return merge(params, (; asymmetry = value))
    error("Unknown parameter field $field.")
end

function clamp_params(target, params)
    bounds = parameter_bounds(target)
    return (
        EJmax = clamp(params.EJmax, bounds.EJmax[1], bounds.EJmax[2]),
        EC = clamp(params.EC, bounds.EC[1], bounds.EC[2]),
        flux = clamp(params.flux, bounds.flux[1], bounds.flux[2]),
        asymmetry = clamp(params.asymmetry, bounds.asymmetry[1], bounds.asymmetry[2]),
    )
end


clamp_params (generic function with 1 method)

In [6]:
function inner_parameter_search(target, flux, asymmetry; keep = 4)
    grids = coarse_grids(target)
    rows = NamedTuple[]

    for EC in grids.EC
        for EJmax in grids.EJmax
            params = (EJmax = EJmax, EC = EC, flux = flux, asymmetry = asymmetry)
            candidate = evaluate_candidate(target, params; include_slope = false)
            update_topk!(rows, candidate; k = keep)
        end
    end

    return rows
end

function coordinate_descent(target, seed)
    steps = if target.kind == :qubit
        Dict(
            :EJmax => [0.10, 0.05, 0.02],
            :EC => [0.0020, 0.0010, 0.0005],
            :flux => [0.010, 0.005, 0.002],
            :asymmetry => [0.050, 0.020, 0.010],
        )
    else
        Dict(
            :EJmax => [1.00, 0.50, 0.25],
            :EC => [0.0025, 0.0010, 0.0005],
            :flux => [0.010, 0.005, 0.002],
            :asymmetry => [0.050, 0.020, 0.010],
        )
    end

    best = evaluate_candidate(target, clamp_params(target, seed); include_slope = true)

    for stage in eachindex(steps[:EJmax])
        improved = true
        while improved
            improved = false
            params = base_params(best)
            for field in (:EJmax, :EC, :flux, :asymmetry)
                for direction in (-1.0, 1.0)
                    trial_value = getfield(params, field) + direction * steps[field][stage]
                    trial_params = clamp_params(target, set_param(params, field, trial_value))
                    trial = evaluate_candidate(target, trial_params; include_slope = true)
                    if trial.loss + 1e-10 < best.loss
                        best = trial
                        improved = true
                        params = base_params(best)
                    end
                end
            end
        end
    end

    return best
end

function deduplicate_rows(rows; keep = 5)
    unique_rows = NamedTuple[]
    seen = Set{NTuple{4, Int}}()

    for row in sort(rows, by = item -> item.loss)
        key = (
            round(Int, row.EJmax * 10_000),
            round(Int, row.EC * 1_000_000),
            round(Int, row.flux * 10_000),
            round(Int, row.asymmetry * 10_000),
        )
        key in seen && continue
        push!(seen, key)
        push!(unique_rows, row)
        length(unique_rows) >= keep && break
    end

    return unique_rows
end

function run_inference(target; seeds_per_slice = 3, multistart = 8, keep = 5)
    grids = coarse_grids(target)
    seeds = NamedTuple[]

    for flux in grids.flux
        for asymmetry in grids.asymmetry
            for seed in inner_parameter_search(target, flux, asymmetry; keep = seeds_per_slice)
                update_topk!(seeds, seed; k = 20)
            end
        end
    end

    refined = NamedTuple[]
    for seed in first(seeds, min(multistart, length(seeds)))
        update_topk!(refined, coordinate_descent(target, base_params(seed)); k = keep * 3)
    end

    return deduplicate_rows(refined; keep = keep)
end

function rounded_rows(rows)
    return [
        (
            rank = index,
            loss = round(row.loss, digits = 4),
            EJmax = round(row.EJmax, digits = 4),
            EC = round(row.EC, digits = 6),
            flux = round(row.flux, digits = 4),
            asymmetry = round(row.asymmetry, digits = 4),
            f01 = round(row.f01, digits = 5),
            alpha = round(row.alpha, digits = 5),
            effEJ = round(row.effEJ, digits = 4),
            slope = round(row.slope, digits = 6),
        ) for (index, row) in enumerate(rows)
    ]
end

function residual_report(target, row)
    if target.kind == :qubit
        return (
            f01_residual_mhz = round(1000 * (row.f01 - target.target_f01), digits = 2),
            EC_residual_mhz = round(1000 * (row.EC - target.target_EC), digits = 2),
            effEJ_residual_mhz = round(1000 * (row.effEJ - target.target_EJ), digits = 2),
            alpha_ghz = round(row.alpha, digits = 5),
            slope = round(row.slope, digits = 6),
        )
    end

    return (
        alpha_residual_mhz = round(1000 * (row.alpha - target.target_alpha), digits = 2),
        plausible_f01_window = target.plausible_f01,
        parked_f01_ghz = round(row.f01, digits = 5),
        slope = round(row.slope, digits = 6),
    )
end

function parameter_spans(rows)
    return (
        EJmax = maximum(getfield.(rows, :EJmax)) - minimum(getfield.(rows, :EJmax)),
        EC = maximum(getfield.(rows, :EC)) - minimum(getfield.(rows, :EC)),
        flux = maximum(getfield.(rows, :flux)) - minimum(getfield.(rows, :flux)),
        asymmetry = maximum(getfield.(rows, :asymmetry)) - minimum(getfield.(rows, :asymmetry)),
    )
end

function fit_classification(target, rows)
    spans = parameter_spans(rows)
    max_abs_flux = maximum(abs.(getfield.(rows, :flux)))

    if target.kind == :qubit && spans.EJmax <= 0.20 && spans.EC <= 0.004 && max_abs_flux <= 0.01
        return "identified"
    elseif target.kind == :coupler && spans.EJmax >= 2.0
        return "weakly identified"
    end

    return "weakly identified"
end


fit_classification (generic function with 1 method)

## Step 4 - Qubit fits

The qubit branches have three direct local anchors: `f01`, derived `EC`, and a parked `EJ` prior from Table II. That is enough to freeze a practical local prior set even though asymmetry itself stays weakly constrained at the sweet spot.


In [7]:
qb1_rows = run_inference(paper_targets[:QB1])
qb2_rows = run_inference(paper_targets[:QB2])

qubit_tables = Dict(
    :QB1 => rounded_rows(qb1_rows),
    :QB2 => rounded_rows(qb2_rows),
)

qubit_residuals = Dict(
    :QB1 => residual_report(paper_targets[:QB1], qb1_rows[1]),
    :QB2 => residual_report(paper_targets[:QB2], qb2_rows[1]),
)

qubit_classification = Dict(
    :QB1 => fit_classification(paper_targets[:QB1], qb1_rows),
    :QB2 => fit_classification(paper_targets[:QB2], qb2_rows),
)

(qubit_tables, qubit_residuals, qubit_classification)


(Dict(:QB2 => [(rank = 1, loss = 0.5643, EJmax = 13.65, EC = 0.200581, flux = 0.0, asymmetry = 0.1, f01 = 4.46979, alpha = -0.22492, effEJ = 13.65, slope = 0.0), (rank = 2, loss = 1.0308, EJmax = 13.78, EC = 0.198581, flux = 0.0, asymmetry = 0.1, f01 = 4.47072, alpha = -0.22238, effEJ = 13.78, slope = 0.0)], :QB1 => [(rank = 1, loss = 0.3775, EJmax = 14.92, EC = 0.199461, flux = 0.0, asymmetry = 0.1, f01 = 4.67066, alpha = -0.22225, effEJ = 14.92, slope = 0.0), (rank = 2, loss = 1.0115, EJmax = 14.78, EC = 0.201461, flux = 0.0, asymmetry = 0.1, f01 = 4.66981, alpha = -0.22477, effEJ = 14.78, slope = 0.0)]), Dict(:QB2 => (f01_residual_mhz = -0.21, EC_residual_mhz = 0.0, effEJ_residual_mhz = -150.0, alpha_ghz = -0.22492, slope = 0.0), :QB1 => (f01_residual_mhz = 0.66, EC_residual_mhz = 0.0, effEJ_residual_mhz = 120.0, alpha_ghz = -0.22225, slope = 0.0)), Dict(:QB2 => "identified", :QB1 => "identified"))

## Step 5 - Coupler local priors

The coupler problem is intentionally weaker. This notebook only asks for a local prior that matches `alpha_c ~= -0.11 GHz`, sits at the sweet spot, and lands in a plausible parked-frequency window. It does not try to uniquely reconstruct the physical coupler.


In [8]:
tc1_rows = run_inference(paper_targets[:TC1])
tc2_rows = run_inference(paper_targets[:TC2])

coupler_tables = Dict(
    :TC1 => rounded_rows(tc1_rows),
    :TC2 => rounded_rows(tc2_rows),
)

coupler_residuals = Dict(
    :TC1 => residual_report(paper_targets[:TC1], tc1_rows[1]),
    :TC2 => residual_report(paper_targets[:TC2], tc2_rows[1]),
)

coupler_classification = Dict(
    :TC1 => fit_classification(paper_targets[:TC1], tc1_rows),
    :TC2 => fit_classification(paper_targets[:TC2], tc2_rows),
)

(coupler_tables, coupler_residuals, coupler_classification)


(Dict(:TC1 => [(rank = 1, loss = 0.0, EJmax = 34.25, EC = 0.105, flux = 0.0, asymmetry = 0.1, f01 = 5.25661, alpha = -0.10999, effEJ = 34.25, slope = 0.0)], :TC2 => [(rank = 1, loss = 0.0, EJmax = 34.25, EC = 0.105, flux = 0.0, asymmetry = 0.1, f01 = 5.25661, alpha = -0.10999, effEJ = 34.25, slope = 0.0)]), Dict(:TC1 => (alpha_residual_mhz = 0.01, plausible_f01_window = (5.0, 8.5), parked_f01_ghz = 5.25661, slope = 0.0), :TC2 => (alpha_residual_mhz = 0.01, plausible_f01_window = (5.0, 8.5), parked_f01_ghz = 5.25661, slope = 0.0)), Dict(:TC1 => "weakly identified", :TC2 => "weakly identified"))

## Step 6 - Identifiability and profile scans

The top-5 tables show where the fit is tight and where it is flat. The profile scans below make the flat directions explicit:

- qubits: asymmetry is nearly flat around the sweet spot once `f01` and `EC` are matched
- couplers: a broad `EJmax` band survives because `alpha` mostly pins `EC`, not the full SQUID parameter set


In [9]:
function profile_scan(target, row, field::Symbol, values)
    params0 = base_params(row)
    return [
        begin
            trial = evaluate_candidate(target, set_param(params0, field, value); include_slope = true)
            (; value, loss = trial.loss)
        end for value in values
    ]
end

qb1_asym_scan = profile_scan(paper_targets[:QB1], qb1_rows[1], :asymmetry, 0.0:0.01:0.20)
qb2_asym_scan = profile_scan(paper_targets[:QB2], qb2_rows[1], :asymmetry, 0.0:0.01:0.20)
tc1_ej_scan = profile_scan(paper_targets[:TC1], tc1_rows[1], :EJmax, 30.0:0.5:40.0)
tc2_ej_scan = profile_scan(paper_targets[:TC2], tc2_rows[1], :EJmax, 30.0:0.5:40.0)

profile_plots = (
    qb1_asymmetry = lineplot([row.value for row in qb1_asym_scan], [row.loss for row in qb1_asym_scan]; title = "QB1 asymmetry profile", xlabel = "asymmetry", ylabel = "loss"),
    qb2_asymmetry = lineplot([row.value for row in qb2_asym_scan], [row.loss for row in qb2_asym_scan]; title = "QB2 asymmetry profile", xlabel = "asymmetry", ylabel = "loss"),
    tc1_ejmax = lineplot([row.value for row in tc1_ej_scan], [row.loss for row in tc1_ej_scan]; title = "TC1 EJmax profile", xlabel = "EJmax (GHz)", ylabel = "loss"),
    tc2_ejmax = lineplot([row.value for row in tc2_ej_scan], [row.loss for row in tc2_ej_scan]; title = "TC2 EJmax profile", xlabel = "EJmax (GHz)", ylabel = "loss"),
)

profile_plots


(qb1_asymmetry =           ⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀QB1 asymmetry profile⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀ 
          ┌────────────────────────────────────────┐ 
        2 │⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠│ 
          │⠘⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠃│ 
          │⠀⠸⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠇⠀│ 
          │⠀⠀⠘⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠎⠀⠀│ 
          │⠀⠀⠀⠸⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠎⠀⠀⠀│ 
          │⠀⠀⠀⠀⠘⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⠃⠀⠀⠀⠀│ 
          │⠀⠀⠀⠀⠀⠘⢆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡰⠃⠀⠀⠀⠀⠀│ 
   loss   │⠀⠀⠀⠀⠀⠀⠈⢢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠜⠀⠀⠀⠀⠀⠀⠀│ 
          │⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⡠⠊⠀⠀⠀⠀⠀⠀⠀⠀│ 
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠣⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠜⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ 
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠒⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠔⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ 
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⠢⢀⣀⠀⠀⠀⠀⣀⡠⠤⠒⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ 
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⠉⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ 
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ 
        0 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ 
          └────────────────────────────────────────┘ 
          ⠀

## Step 7 - Freeze `paper_local_priors_v1`

The accepted snapshot is written directly to `output/renger2026/paper_local_priors.toml`. `TC1` and `TC2` remain intentionally identical here because local single-device information does not distinguish them yet.


In [10]:
function snapshot_device(target, row, classification)
    payload = Dict{String, Any}(
        "type" => target.kind == :qubit ? "TunableTransmon" : "TunableCoupler",
        "EJmax" => round(row.EJmax, digits = 6),
        "EC" => round(row.EC, digits = 12),
        "flux" => round(row.flux, digits = 6),
        "asymmetry" => round(row.asymmetry, digits = 6),
        "ng" => 0.0,
        "ncut" => target.ncut,
        "charge_cutoff" => target.charge_cutoff,
        "fit_classification" => classification,
        "f01_ghz" => round(row.f01, digits = 5),
        "anharmonicity_ghz" => round(row.alpha, digits = 5),
    )

    if target.kind == :qubit
        payload["residual_f01_mhz"] = round(1000 * (row.f01 - target.target_f01), digits = 2)
    else
        payload["residual_alpha_mhz"] = round(1000 * (row.alpha - target.target_alpha), digits = 2)
    end

    return payload
end

accepted_rows = Dict(
    :QB1 => qb1_rows[1],
    :QB2 => qb2_rows[1],
    :TC1 => tc1_rows[1],
    :TC2 => tc2_rows[1],
)

accepted_snapshot = Dict{String, Any}(
    "schema" => "paper_local_priors_v1",
    "paper" => "Renger et al. (2026)",
    "source_notebook" => "output/jupyter-notebook/renger2026-parameter-inference.ipynb",
    "scope" => "Local single-device priors for QB1-TC1-CR-TC2-QB2",
    "notes" => "Qubit fits are frozen local priors. Coupler fits are weakly identified and intended only as initialization for later coupled-model fitting.",
    "targets" => Dict(
        "qb1_f01_ghz" => paper_targets[:QB1].target_f01,
        "qb1_ej_ghz" => paper_targets[:QB1].target_EJ,
        "qb1_ej_over_ec" => 74.2,
        "qb1_ec_ghz" => paper_targets[:QB1].target_EC,
        "qb2_f01_ghz" => paper_targets[:QB2].target_f01,
        "qb2_ej_ghz" => paper_targets[:QB2].target_EJ,
        "qb2_ej_over_ec" => 68.8,
        "qb2_ec_ghz" => paper_targets[:QB2].target_EC,
        "cr_f01_ghz" => secondary_constraints.resonator_f01,
        "coupler_alpha_prior_ghz" => secondary_constraints.coupler_alpha_prior,
        "beta_qc_qb1" => secondary_constraints.beta_qc_qb1,
        "beta_qc_qb2" => secondary_constraints.beta_qc_qb2,
        "beta_cr" => secondary_constraints.beta_cr,
        "beta_qr" => secondary_constraints.beta_qr,
    ),
    "inference" => Dict(
        "hamiltonian_spec" => "CircuitHamiltonianSpec(charge_cutoff = 10)",
        "sweet_spot_flux_target" => 0.0,
        "asymmetry_prior_center" => 0.10,
        "qubit_fit_classification" => "identified",
        "coupler_fit_classification" => "weakly identified",
    ),
    "devices" => Dict(
        "QB1" => snapshot_device(paper_targets[:QB1], accepted_rows[:QB1], qubit_classification[:QB1]),
        "QB2" => snapshot_device(paper_targets[:QB2], accepted_rows[:QB2], qubit_classification[:QB2]),
        "TC1" => snapshot_device(paper_targets[:TC1], accepted_rows[:TC1], coupler_classification[:TC1]),
        "TC2" => snapshot_device(paper_targets[:TC2], accepted_rows[:TC2], coupler_classification[:TC2]),
    ),
)

output_dir = joinpath(project_root, "output", "renger2026")
mkpath(output_dir)
snapshot_path = joinpath(output_dir, "paper_local_priors.toml")
open(snapshot_path, "w") do io
    TOML.print(io, accepted_snapshot)
end

(snapshot_path, accepted_snapshot["devices"])


("/Users/yalgaeahn/Research/20_Projects/QuantumCircuit.jl/output/renger2026/paper_local_priors.toml", Dict{String, Dict{String, Any}}("QB2" => Dict("ncut" => 15, "residual_f01_mhz" => -0.21, "anharmonicity_ghz" => -0.22492, "EC" => 0.200581395349, "EJmax" => 13.65, "flux" => 0.0, "ng" => 0.0, "charge_cutoff" => 10, "asymmetry" => 0.1, "type" => "TunableTransmon"…), "TC2" => Dict("ncut" => 13, "residual_alpha_mhz" => 0.01, "anharmonicity_ghz" => -0.10999, "EC" => 0.105, "EJmax" => 34.25, "flux" => 0.0, "ng" => 0.0, "charge_cutoff" => 10, "asymmetry" => 0.1, "type" => "TunableCoupler"…), "TC1" => Dict("ncut" => 13, "residual_alpha_mhz" => 0.01, "anharmonicity_ghz" => -0.10999, "EC" => 0.105, "EJmax" => 34.25, "flux" => 0.0, "ng" => 0.0, "charge_cutoff" => 10, "asymmetry" => 0.1, "type" => "TunableCoupler"…), "QB1" => Dict("ncut" => 15, "residual_f01_mhz" => 0.66, "anharmonicity_ghz" => -0.22225, "EC" => 0.199460916442, "EJmax" => 14.92, "flux" => 0.0, "ng" => 0.0, "charge_cutoff" => 10, 

## Step 8 - Flux-control sanity checks on uncoupled devices

These checks stay inside the current supported surface:

- one local tunable subsystem at a time
- circuit Hamiltonian only
- no coupled-device pulse calibration yet

The two tests here are minimal but useful:

- a constant `FluxControl` shift should match the corresponding static shifted model
- a small sinusoidal pulse should produce nontrivial but stable population transfer


In [11]:
kwarg(symbol::Symbol, value) = NamedTuple{(symbol,)}((value,))

function flux_sanity_check(label, target, row; delta_flux = 0.015, pulse_delta = 0.025, pulse_omega = 4.5)
    params = base_params(row)
    device = build_local_device(target, params)
    system = CompositeSystem(device)
    spec = circuit_spec(target)
    ψ0 = basis_state(system; hamiltonian_spec = spec, kwarg(target.name, 0)...)
    t_short = collect(range(0.0, 4.0; length = 41))
    t_long = collect(range(0.0, 12.0; length = 121))
    charge_obs = ObservableSpec(Symbol(:charge_, target.name), target.name, :charge)

    shifted_system = with_subsystem_parameter(system, target.name, :flux, params.flux + delta_flux)
    shifted_result = evolve(
        shifted_system,
        basis_state(shifted_system; hamiltonian_spec = spec, kwarg(target.name, 0)...),
        t_short;
        hamiltonian_spec = spec,
        observables = [charge_obs],
    )

    constant_result = evolve(
        system,
        ψ0,
        t_short;
        hamiltonian_spec = spec,
        observables = [charge_obs],
        flux_controls = [FluxControl(Symbol(:constant_, target.name), target.name, (p, t) -> delta_flux)],
    )

    shifted_trace = observable_trace(shifted_result, charge_obs.label).values
    constant_trace = observable_trace(constant_result, charge_obs.label).values
    max_constant_error = maximum(abs.(constant_trace .- shifted_trace))

    pulse = FluxControl(
        Symbol(:pulse_, target.name),
        target.name,
        (p, t) -> p.delta * sin(p.omega * t);
        derivative = (p, t) -> p.delta * p.omega * cos(p.omega * t),
    )
    pulsed_result = evolve(
        system,
        ψ0,
        t_long;
        hamiltonian_spec = spec,
        flux_controls = [pulse],
        params = (; delta = pulse_delta, omega = pulse_omega),
    )
    excited_population = population_trace(pulsed_result, target.name, 1)

    return (
        device = String(label),
        max_constant_shift_error = max_constant_error,
        max_excited_population = maximum(excited_population.values),
        final_excited_population = excited_population.values[end],
    )
end

sanity_summary = [
    flux_sanity_check(:QB1, paper_targets[:QB1], accepted_rows[:QB1]),
    flux_sanity_check(:TC1, paper_targets[:TC1], accepted_rows[:TC1]),
]

sanity_summary


2-element Vector{@NamedTuple{device::String, max_constant_shift_error::Float64, max_excited_population::Float64, final_excited_population::Float64}}:
 (device = "QB1", max_constant_shift_error = 1.1456555538913449e-5, max_excited_population = 0.5334158311181706, final_excited_population = 0.04498041682939048)
 (device = "TC1", max_constant_shift_error = 3.864584450674613e-6, max_excited_population = 0.4143046603429137, final_excited_population = 0.08662523984456122)

## Result

This notebook produces a reproducible local prior set that matches the paper targets as closely as the single-device information allows today.

Practical conclusion:

- `QB1` and `QB2` can be frozen now as local tunable-subsystem priors.
- `TC1` and `TC2` should still be treated as initialization points, not device truth.
- The next fitting stage should use coupled avoided crossings and gate-operating-point data to separate the two coupler branches.
